# Model A – 2D CNN for JWST / Hubble-style transit spectroscopy
**Input:** an image of *wavelength (rows) × time (columns)* – this is what JWST/HST give you during a transit (one light curve per wavelength channel).
**Task:** which molecules (H₂O, CO₂, CH₄, SO₂) leave a signature? (multi-label)

**Training data is simulated** (toy molecular bands + noise) because real JWST/HST spectra are too few to train on. Real data is used for *testing/demo* at the end.
Runs in a few minutes on a T4 GPU; no downloads needed.

In [ ]:
import time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader
dev='cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
TRAIN_BUDGET=15   # minutes

## 1. Simulator (replace with petitRADTRANS / batman later for realism)

In [ ]:
NW, NT = 64, 64
WAVE = np.linspace(1.0, 5.0, NW)
MOLS = ['H2O','CO2','CH4','SO2']
BANDS = {'H2O':[(1.4,.10),(1.9,.12),(2.7,.18)], 'CO2':[(2.0,.08),(2.7,.10),(4.3,.15)],
         'CH4':[(1.7,.08),(2.3,.10),(3.3,.15)], 'SO2':[(4.0,.12)]}
def profile(m):
    p = sum(np.exp(-0.5*((WAVE-c)/w)**2) for c,w in BANDS[m]); return p/p.max()
PROF = np.stack([profile(m) for m in MOLS])
_t = np.linspace(-1,1,NT)
SHAPE = 0.5*(np.tanh((_t+0.5)/0.06)-np.tanh((_t-0.5)/0.06))
def make_batch(n, rng):
    labels = (rng.random((n,4))<0.5).astype(np.float32)
    amps = rng.uniform(2e-4,1.0e-3,(n,4))*labels
    depth = rng.uniform(0.005,0.02,(n,1)) + amps@PROF
    noise = rng.uniform(1e-3,3e-3,(n,1,1))
    flux = 1 - depth[:,:,None]*SHAPE[None,None,:] + rng.normal(size=(n,NW,NT))*noise
    img = flux-1
    scale = -img[:,:,24:40].mean(axis=(1,2),keepdims=True)
    img = (img/scale).astype(np.float32)
    return img[:,None], labels

In [ ]:
rng=np.random.default_rng(0)
Xtr,ytr=make_batch(8000,rng); Xva,yva=make_batch(1000,rng); Xte,yte=make_batch(1000,rng)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].imshow(Xtr[0,0],aspect='auto',extent=[-1,1,5,1]); ax[0].set_title(f'Example image, labels={ytr[0]}'); ax[0].set_ylabel('wavelength (µm)'); ax[0].set_xlabel('time')
ax[1].plot(WAVE,Xtr[0,0,:,24:40].mean(-1)); ax[1].set_title('its in-transit spectrum'); plt.show()
mk=lambda X,y,sh: DataLoader(TensorDataset(torch.tensor(X),torch.tensor(y)),batch_size=64,shuffle=sh)
dl_tr,dl_va,dl_te=mk(Xtr,ytr,True),mk(Xva,yva,False),mk(Xte,yte,False)

## 2. The 2D CNN

In [ ]:
class SpecCNN2D(nn.Module):
    def __init__(s):
        super().__init__()
        blk=lambda i,o: nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(),nn.MaxPool2d(2))
        s.features=nn.Sequential(blk(1,16),blk(16,32),blk(32,64),nn.Conv2d(64,64,3,padding=1),nn.ReLU())  # last conv = Grad-CAM layer
        s.head=nn.Sequential(nn.Dropout(.3),nn.Linear(64,4))        # 4 molecules
    def forward(s,x): return s.head(s.features(x).mean((-2,-1)))
model=SpecCNN2D().to(dev); opt=torch.optim.AdamW(model.parameters(),1e-3,weight_decay=1e-2); lossf=nn.BCEWithLogitsLoss()

## 3. Train

In [ ]:
def evaluate(dl):
    model.eval(); P=[];Y=[]
    with torch.no_grad():
        for xb,yb in dl: P.append(torch.sigmoid(model(xb.to(dev))).cpu().numpy()); Y.append(yb.numpy())
    P,Y=np.concatenate(P),np.concatenate(Y); return [roc_auc_score(Y[:,i],P[:,i]) for i in range(4)]
best,t0=0,time.time()
for ep in range(60):
    model.train()
    for xb,yb in dl_tr:
        xb,yb=xb.to(dev),yb.to(dev); opt.zero_grad(); loss=lossf(model(xb),yb); loss.backward(); opt.step()
    aucs=evaluate(dl_va); m=np.mean(aucs); print(f'ep {ep:02d} loss {loss.item():.3f} mean val AUC {m:.3f}')
    if m>best: best=m; torch.save(model.state_dict(),'model_a_2d.pt')
    if (time.time()-t0)/60>TRAIN_BUDGET: print('time budget reached'); break
model.load_state_dict(torch.load('model_a_2d.pt'))
print(dict(zip(MOLS,np.round(evaluate(dl_te),3))))

## 4. Grad-CAM in 2D: *which wavelengths and times* drove the prediction?

In [ ]:
def gradcam2d(x,k):
    model.eval(); f=model.features(x[None].to(dev)); f.retain_grad(); out=model.head(f.mean((-2,-1)))[0,k]; out.backward()
    cam=F.relu((f.grad.mean((-2,-1),keepdim=True)*f).sum(1,keepdim=True))
    cam=F.interpolate(cam,size=(NW,NT),mode='bilinear')[0,0].detach().cpu().numpy(); return cam/(cam.max()+1e-8), torch.sigmoid(out).item()
i=int(np.where(yte[:,0]==1)[0][0]); k=0                    # first test example that truly contains H2O
cam,p=gradcam2d(torch.tensor(Xte[i]),k)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].imshow(Xte[i,0],aspect='auto',extent=[-1,1,5,1]); ax[0].set_title('input')
ax[1].imshow(cam,aspect='auto',extent=[-1,1,5,1],cmap='inferno'); ax[1].set_title(f'Grad-CAM for {MOLS[k]} (prob {p:.2f})')
for a in ax: a.set_ylabel('wavelength (µm)'); a.set_xlabel('time')
plt.show()

## 5. Using it on REAL data (when you have it)
You need a reduced **wavelength × time flux matrix** for one transit (from Eureka!/jwst pipeline output, or an HST WFC3 spectroscopic light-curve product). Then:

In [ ]:
from scipy.ndimage import zoom
def prepare_real(flux, wave, time):
    # flux: (n_wave, n_time) normalised to ~1 out of transit. Resample to the 64x64 grid the model knows.
    keep=(wave>=1.0)&(wave<=5.0); flux=flux[keep]
    img=zoom(flux-1,(NW/flux.shape[0],NT/flux.shape[1]),order=1)
    img=img/(-img[:,24:40].mean())
    return torch.tensor(img[None].astype(np.float32))
def predict(img):
    model.eval()
    with torch.no_grad(): return dict(zip(MOLS,torch.sigmoid(model(img[None].to(dev)))[0].cpu().numpy().round(2)))
# predict(prepare_real(my_flux, my_wave, my_time))

**Caveat to tell judges honestly:** the model learned from simulated, toy band positions, so its output on real data is a *screening hint*, not a detection. Confirm with a proper atmospheric retrieval.